In [1]:
using ITensors
using ITensorMPS

using Random

#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end
function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    # Random order of site indices
    up_sites = shuffle(1:L)[1:Nup]
    for i in up_sites
        state[i] = "Up"
    end
    dn_sites = shuffle(1:L)
    added_dn = 0
    for i in dn_sites
        if added_dn == Ndn
            break
        end
        if state[i] == "Emp"
            state[i] = "Dn"
        elseif state[i] == "Up"
            state[i] = "UpDn"
        end
        added_dn += 1
    end
    return state
end

function random_ps_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)

    println("floor L / 2 = ", floor(Int, (L/2)))

    for i in 1:floor(Int, (L/2)) #  - Nup_extra)
        println("i = $i")
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[floor(Int, L / 2)] = "Up"
    end
    return state
end 

function random_ps_state(L, Nup, Ndn)
    state = fill("Emp", L)
    
    Ndbl = min(Nup, Ndn)
    Nup -= Ndbl
    Ndn -= Ndbl
    
    for i in 1:Ndbl
        state[i] = "UpDn"
    end
    
    next_site = Ndbl + 1
    
    if Nup > 0
        state[next_site] = "Up"
        next_site += 1
    elseif Ndn > 0
        state[next_site] = "Dn"
        next_site += 1
    end
    return state
end


using Statistics
function average_single_site_entanglement(L, up, dn, updn)
    single_site_entanglement = fill(0.0, L)

    for i in 1:L 
        w_2 = updn[i] 
        w_up = up[i] - w_2 
        w_dn = dn[i] - w_2 
        w_0 = 1 - w_up - w_dn - w_2 
        single_site_entanglement[i] = 1 - (w_2^2 + w_up^2 + w_dn^2 + w_0^2)
    end
    return Statistics.mean(single_site_entanglement)
end 
#=
    Returns the density of up and down 
    electrons.
=#
function density_operators(N, psi)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
    orthogonalize!(psi, j)
    psidag_j = dag(prime(psi[j], "Site"))
    upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
    dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
    updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

density_operators (generic function with 1 method)

In [9]:
# Trying to use the MPS form to compute the reduced density matrix 
# properties. 
# For now comparing it to the single-site entanglement computed from correlators.
function average_entanglement_entropy(psi, L)
    # Average entropy taken over all sites. 
    S_avg = 0.0
    # Over all sites j
    for j = 1:L 

        # change orthogonality center to j
        psi = orthogonalize(psi, j)

        # tensor at site j
        A = psi[j]

        # prime the physical index 
        A_dag = dag(A)
        prime!(A_dag, "Site")

        rdm = A * A_dag
        # Diagonalize
        D, U = eigen(rdm)

        # Compute von Neumann entropy safely
        S = 0.0
        for n=1:dim(D, 1)
            p = D[n,n] 
            p_real = real(p)
            if p_real > 1e-12
                S -= p_real * log(p_real)
            end
        end
        S_avg += S 
    end
    return S_avg / L 
end

average_entanglement_entropy (generic function with 1 method)

In [6]:
L = 5

sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 6

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

state = random_metallic_state(L, Nup, Ndn)

psi0 = random_mps(sites, state; linkdims=10)

U = 0.0 
V = 0.0
J = 1.0
H = H_EHM(L, J, U, V, sites)

energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

After sweep 1 energy=-5.463579822192987  maxlinkdim=16 maxerr=0.00E+00 time=14.887
After sweep 2 energy=-5.464101603369136  maxlinkdim=16 maxerr=0.00E+00 time=0.028
After sweep 3 energy=-5.464101615137744  maxlinkdim=16 maxerr=0.00E+00 time=0.044
After sweep 4 energy=-5.464101615137758  maxlinkdim=16 maxerr=0.00E+00 time=0.038
After sweep 5 energy=-5.464101615137757  maxlinkdim=16 maxerr=0.00E+00 time=0.036
After sweep 6 energy=-5.464101615137757  maxlinkdim=16 maxerr=0.00E+00 time=0.055


(-5.464101615137757, MPS
[1] ((dim=4|id=962|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=426|"Link,l=1") <Out>
 1: QN(("Nf",3,-1),("Sz",1)) => 1
 2: QN(("Nf",4,-1),("Sz",0)) => 1
 3: QN(("Nf",4,-1),("Sz",2)) => 1
 4: QN(("Nf",5,-1),("Sz",1)) => 1)
[2] ((dim=16|id=790|"Link,l=2") <Out>
 1: QN(("Nf",1,-1),("Sz",1)) => 1
 2: QN(("Nf",2,-1),("Sz",0)) => 2
 3: QN(("Nf",2,-1),("Sz",2)) => 2
 4: QN(("Nf",3,-1),("Sz",-1)) => 1
 5: QN(("Nf",3,-1),("Sz",1)) => 4
 6: QN(("Nf",3,-1),("Sz",3)) => 1
 7: QN(("Nf",4,-1),("Sz",0)) => 2
 8: QN(("Nf",4,-1),("Sz",2)) => 2
 9: QN(("Nf",5,-1),("Sz",1)) => 1, (dim=4|id=240|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=426|"Link,l=1") <In>
 1: QN(("Nf",3,-1),("Sz",1)) => 1
 2: QN(("Nf",4,-1),("Sz",0)) => 

Resultado por ED: $\mathcal{L} = 0.7148148148148148$


In [7]:
println("lenght psi = ",  length(psi))

lenght psi = 5


In [10]:
E_p = average_site_entropy_manual(psi, L)
println("E_p = ", E_p)

upd, dnd, updn = density_operators(L, psi)
S = average_single_site_entanglement(L, upd, dnd, updn)

println("S = ", S)

E_p = 22.78575494699123
S = 0.7148148148148146
